# 10 — Painel de Report

**Objetivo:** preparar as tabelas resumo que alimentam o dashboard executivo e técnico em AI/BI Dashboards.

**Conteúdo do painel:**

| Seção | Indicadores |
| --- | --- |
| Visão executiva | Clientes por snapshot, taxa de recompra, AUC do campeão |
| Performance dos modelos | AUC/KS/Brier — validação vs. teste (decaimento) |
| Segmentação | Distribuição de scores por decisão, lift e ganhos acumulados |
| Interpretabilidade | Importância de variáveis do campeão |
| Estabilidade | AUC por snapshot (drift de discriminação) |
| Campanha | Simulação: quantos clientes capturar em cada faixa de score |

**Output:** tabelas `painel_*` no schema `ml_bakehouse`.

In [0]:
# ── Configuração central (idem notebook 00) ──
CATALOG = "workspace"
SCHEMA  = "ml_bakehouse"
CAMPEAO = None   # None = pega automaticamente o campeão do model_evaluation

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window

model_eval = spark.table(f"{CATALOG}.{SCHEMA}.model_evaluation").toPandas()
if CAMPEAO is None:
    CAMPEAO = model_eval.sort_values("auc_teste", ascending=False).iloc[0]["modelo"]
print("Modelo campeão exibido no painel:", CAMPEAO)

preds = spark.table(f"{CATALOG}.{SCHEMA}.model_predictions")

In [0]:
# 1) Visão executiva: clientes e taxa de recompra por snapshot + score médio do campeão
test_preds = preds.filter((F.col("base") == "teste") & (F.col("modelo") == CAMPEAO))

executivo = (
    preds.filter(F.col("base") == "validacao")
    .groupBy("reference_date")
    .agg(
        F.count("*").alias("clientes"),
        F.round(F.avg("target"), 4).alias("taxa_recompra"),
    )
    .orderBy("reference_date")
)
display(executivo)

(executivo.write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.painel_resumo_executivo"))

In [0]:
# 2) Scores do campeão por cliente (último snapshot disponível), com decisão de score
w = Window.partitionBy("base", "modelo").orderBy(F.desc("score"))
scores = (
    preds.filter(F.col("modelo") == CAMPEAO)
    .withColumn("decis_score", (F.ntile(10).over(w)).cast("int"))   # 1 = melhores scores
    .withColumn("data_snapshot", F.col("reference_date").cast("string"))
)

(scores.write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.painel_scores"))

# Lift por decisão (teste) para o dashboard
camp = scores.filter(F.col("base") == "teste").toPandas()
taxa_global = camp["target"].mean()
lift = (
    camp.groupby("decis_score")
    .agg(n=("target", "count"), eventos=("target", "sum"), score_medio=("score", "mean"))
    .reset_index()
)
lift["taxa_evento"] = (lift["eventos"] / lift["n"]).round(4)
lift["lift"] = (lift["taxa_evento"] / taxa_global).round(2)

(spark.createDataFrame(lift)
     .write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.painel_decis_lift"))
display(lift.sort_values("decis_score"))

# Simulação de campanha: captura de eventos acumulada por volume acionado
camp_sorted = camp.sort_values("score", ascending=False).reset_index(drop=True)
campanha = pd.DataFrame({
    "pct_clientes_acionados": [round(p, 2) for p in (camp_sorted.index + 1) / len(camp_sorted)],
})
campanha["eventos_capturados_%"] = (camp_sorted["target"].cumsum() / camp_sorted["target"].sum() * 100).round(1)
(spark.createDataFrame(campanha)
     .write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.painel_curva_ganhos"))
display(spark.table(f"{CATALOG}.{SCHEMA}.painel_curva_ganhos").filter("pct_clientes_acionados <= 0.5"))

In [0]:
# 3) Performance dos modelos e evolução temporal (para o dashboard)
# model_evaluation já contém a comparação de modelos — exposta como visão
spark.sql(f"""
  CREATE OR REPLACE VIEW {CATALOG}.{SCHEMA}.painel_comparacao_modelos AS
  SELECT * FROM {CATALOG}.{SCHEMA}.model_evaluation
""")
display(spark.table(f"{CATALOG}.{SCHEMA}.painel_comparacao_modelos"))

# AUC por snapshot do campeão (drift de discriminação)
from sklearn.metrics import roc_auc_score

evol = []
for (data,), g in preds.filter(F.col("modelo") == CAMPEAO).toPandas().groupby(["reference_date"]):
    if g["target"].nunique() > 1:
        evol.append({"reference_date": str(data), "auc": round(roc_auc_score(g["target"], g["score"]), 4)})

(spark.createDataFrame(evol)
     .write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.painel_evolucao_auc"))
display(spark.table(f"{CATALOG}.{SCHEMA}.painel_evolucao_auc").orderBy("reference_date"))

## Como montar o dashboard

As tabelas `painel_*` estão prontas como datasets de um **AI/BI Dashboard**:

1. No workspace: **Dashboards → Create dashboard**
2. Adicione datasets apontando para `workspace.ml_bakehouse.painel_*` e `model_evaluation`
3. Widgets sugeridos:
   * KPI cards: taxa de recompra média, AUC do campeão, lift do 1º decisão
   * Linha: `painel_resumo_executivo` (clientes/taxa ao longo do tempo) e `painel_evolucao_auc`
   * Barra: `painel_decis_lift` (lift por decisão) e `painel_comparacao_modelos`
   * Tabela: `painel_scores` com filtro por decisão (fila de campanha)
4. Publique e compartilhe com as áreas de negócio

> Peça ao assistente: **"crie o dashboard a partir das tabelas painel_*"** — ele monta o rascunho automaticamente.

## Monitoramento e operacionalização

* **Drift:** recompute PSI (05_eda_ii) e AUC por snapshot periodicamente
* **Retreino:** agendar notebooks 01→09 em um Job com frequência semanal
* **Serving:** registrar o campeão no Model Registry e publicar em um Model Serving endpoint para scores em tempo real